In [2]:
# ============================================================
# CLOUD-BASED DISTRIBUTED TWITTER SENTIMENT ANALYSIS
# Phase 1: Data Processing + Machine Learning
# Technology: Apache Spark + Spark MLlib
# ============================================================

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col,
    lower,
    regexp_replace,
    trim,
    length,
    count,
    when
)

from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    StringIndexer,
    Tokenizer,
    StopWordsRemover,
    HashingTF,
    IDF
)

from pyspark.ml.classification import (
    LogisticRegression,
    NaiveBayes,
    RandomForestClassifier
)

from pyspark.ml.evaluation import MulticlassClassificationEvaluator


# ============================================================
# 1. CREATE SPARK SESSION
# ============================================================

spark = SparkSession.builder \
    .appName("CloudTwitterSentimentAnalysis") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

print("\n========================================")
print(" TWITTER SENTIMENT ANALYSIS")
print("========================================")


# ============================================================
# 2. LOAD DATASET
# ============================================================

print("\n[1] Loading dataset...")

df = spark.read.csv(
    "Tweets.csv",
    header=True,
    inferSchema=True
)

print("\nDataset Schema:")
df.printSchema()

print("\nFirst 5 rows:")
df.show(5, truncate=False)

print("\nNumber of rows:", df.count())


# ============================================================
# 3. CHECK REQUIRED COLUMNS
# ============================================================

required_columns = ["text", "sentiment"]

for column_name in required_columns:

    if column_name not in df.columns:
        raise ValueError(
            f"Required column '{column_name}' "
            f"was not found in the dataset."
        )

print("\nRequired columns found successfully.")


# ============================================================
# 4. CHECK MISSING VALUES
# ============================================================

print("\n[2] Checking missing values...")

missing_text = df.filter(
    col("text").isNull()
).count()

missing_sentiment = df.filter(
    col("sentiment").isNull()
).count()

print("Missing text:", missing_text)
print("Missing sentiment:", missing_sentiment)


# ============================================================
# 5. REMOVE NULL VALUES
# ============================================================

df_clean = df.dropna(
    subset=["text", "sentiment"]
)

print(
    "\nRows after removing missing values:",
    df_clean.count()
)


# ============================================================
# 6. REMOVE EMPTY TEXT
# ============================================================

df_clean = df_clean.filter(
    length(trim(col("text"))) > 0
)

print(
    "Rows after removing empty tweets:",
    df_clean.count()
)


# ============================================================
# 7. CHECK DUPLICATES
# ============================================================

duplicate_count = (
    df_clean.count()
    - df_clean.dropDuplicates(["text"]).count()
)

print(
    "\nDuplicate tweet count:",
    duplicate_count
)

# Remove duplicate tweets
df_clean = df_clean.dropDuplicates(["text"])

print(
    "Rows after removing duplicates:",
    df_clean.count()
)


# ============================================================
# 8. SENTIMENT DISTRIBUTION
# ============================================================

print("\n[3] Sentiment Distribution:")

df_clean.groupBy(
    "sentiment"
).count().orderBy(
    col("count").desc()
).show()


# ============================================================
# 9. TEXT CLEANING
# ============================================================

print("\n[4] Cleaning tweet text...")

cleaned_df = df_clean.withColumn(
    "cleaned_text",
    lower(col("text"))
)

# Remove URLs
cleaned_df = cleaned_df.withColumn(
    "cleaned_text",
    regexp_replace(
        col("cleaned_text"),
        r"http\S+|www\S+",
        ""
    )
)

# Remove mentions
cleaned_df = cleaned_df.withColumn(
    "cleaned_text",
    regexp_replace(
        col("cleaned_text"),
        r"@\w+",
        ""
    )
)

# Remove hashtags symbol but preserve hashtag word
cleaned_df = cleaned_df.withColumn(
    "cleaned_text",
    regexp_replace(
        col("cleaned_text"),
        r"#",
        ""
    )
)

# Remove non-English characters and punctuation
cleaned_df = cleaned_df.withColumn(
    "cleaned_text",
    regexp_replace(
        col("cleaned_text"),
        r"[^a-zA-Z\s]",
        " "
    )
)

# Remove multiple spaces
cleaned_df = cleaned_df.withColumn(
    "cleaned_text",
    regexp_replace(
        col("cleaned_text"),
        r"\s+",
        " "
    )
)

# Trim whitespace
cleaned_df = cleaned_df.withColumn(
    "cleaned_text",
    trim(col("cleaned_text"))
)

print("\nCleaned examples:")

cleaned_df.select(
    "text",
    "cleaned_text",
    "sentiment"
).show(
    10,
    truncate=False
)


# ============================================================
# 10. REMOVE TEXT THAT BECAME EMPTY
# ============================================================

cleaned_df = cleaned_df.filter(
    length(col("cleaned_text")) > 0
)

print(
    "\nRows after final text cleaning:",
    cleaned_df.count()
)


# ============================================================
# 11. TRAIN / TEST SPLIT
# ============================================================

print("\n[5] Splitting dataset...")

train_df, test_df = cleaned_df.randomSplit(
    [0.8, 0.2],
    seed=42
)

print("Training rows:", train_df.count())
print("Testing rows:", test_df.count())


# ============================================================
# 12. CREATE SPARK ML PREPROCESSING PIPELINE
# ============================================================

print("\n[6] Creating Spark ML pipeline...")


# Convert sentiment labels to numerical labels
indexer = StringIndexer(
    inputCol="sentiment",
    outputCol="label",
    handleInvalid="skip"
)


# Tokenization
tokenizer = Tokenizer(
    inputCol="cleaned_text",
    outputCol="words"
)


# Remove stopwords
remover = StopWordsRemover(
    inputCol="words",
    outputCol="filtered_words"
)


# Convert words to numerical term frequencies
hashing_tf = HashingTF(
    inputCol="filtered_words",
    outputCol="rawFeatures",
    numFeatures=10000
)


# TF-IDF
idf = IDF(
    inputCol="rawFeatures",
    outputCol="features"
)


# Create preprocessing pipeline
preprocessing_pipeline = Pipeline(
    stages=[
        indexer,
        tokenizer,
        remover,
        hashing_tf,
        idf
    ]
)


# ============================================================
# 13. FIT PREPROCESSING ONLY ON TRAINING DATA
# ============================================================

print("\nFitting preprocessing pipeline...")

preprocessing_model = (
    preprocessing_pipeline.fit(train_df)
)


# Transform training and testing data
train_processed = (
    preprocessing_model.transform(train_df)
)

test_processed = (
    preprocessing_model.transform(test_df)
)


print("\nProcessed training data:")
train_processed.select(
    "sentiment",
    "label",
    "cleaned_text",
    "features"
).show(
    5,
    truncate=False
)


# ============================================================
# 14. LOGISTIC REGRESSION
# ============================================================

print("\n========================================")
print(" MODEL 1: LOGISTIC REGRESSION")
print("========================================")

logistic_regression = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    maxIter=20
)

lr_model = logistic_regression.fit(
    train_processed
)

lr_predictions = lr_model.transform(
    test_processed
)


# ============================================================
# 15. NAIVE BAYES
# ============================================================

print("\n========================================")
print(" MODEL 2: NAIVE BAYES")
print("========================================")

naive_bayes = NaiveBayes(
    featuresCol="features",
    labelCol="label"
)

nb_model = naive_bayes.fit(
    train_processed
)

nb_predictions = nb_model.transform(
    test_processed
)


# ============================================================
# 16. RANDOM FOREST
# ============================================================

print("\n========================================")
print(" MODEL 3: RANDOM FOREST")
print("========================================")

random_forest = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    numTrees=50,
    maxDepth=10,
    seed=42
)

rf_model = random_forest.fit(
    train_processed
)

rf_predictions = random_forest_model = random_forest.fit(
    train_processed
).transform(
    test_processed
)


# ============================================================
# 17. MODEL EVALUATION FUNCTION
# ============================================================

def evaluate_model(
    predictions,
    model_name
):

    print("\n----------------------------------------")
    print(model_name)
    print("----------------------------------------")

    evaluator_accuracy = (
        MulticlassClassificationEvaluator(
            labelCol="label",
            predictionCol="prediction",
            metricName="accuracy"
        )
    )

    evaluator_precision = (
        MulticlassClassificationEvaluator(
            labelCol="label",
            predictionCol="prediction",
            metricName="weightedPrecision"
        )
    )

    evaluator_recall = (
        MulticlassClassificationEvaluator(
            labelCol="label",
            predictionCol="prediction",
            metricName="weightedRecall"
        )
    )

    evaluator_f1 = (
        MulticlassClassificationEvaluator(
            labelCol="label",
            predictionCol="prediction",
            metricName="f1"
        )
    )

    accuracy = evaluator_accuracy.evaluate(
        predictions
    )

    precision = evaluator_precision.evaluate(
        predictions
    )

    recall = evaluator_recall.evaluate(
        predictions
    )

    f1 = evaluator_f1.evaluate(
        predictions
    )

    print(
        f"Accuracy  : {accuracy:.4f}"
    )

    print(
        f"Precision : {precision:.4f}"
    )

    print(
        f"Recall    : {recall:.4f}"
    )

    print(
        f"F1 Score  : {f1:.4f}"
    )

    return {
        "model": model_name,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }


# ============================================================
# 18. EVALUATE ALL MODELS
# ============================================================

print("\n========================================")
print(" MODEL EVALUATION")
print("========================================")

lr_results = evaluate_model(
    lr_predictions,
    "Logistic Regression"
)

nb_results = evaluate_model(
    nb_predictions,
    "Naive Bayes"
)

rf_results = evaluate_model(
    rf_predictions,
    "Random Forest"
)


# ============================================================
# 19. COMPARE MODELS
# ============================================================

results = [
    lr_results,
    nb_results,
    rf_results
]

print("\n========================================")
print(" MODEL COMPARISON")
print("========================================")

for result in results:

    print(
        f"{result['model']:<25} "
        f"Accuracy={result['accuracy']:.4f} "
        f"F1={result['f1']:.4f}"
    )


# ============================================================
# 20. SELECT BEST MODEL
# ============================================================

best_result = max(
    results,
    key=lambda x: x["f1"]
)

print("\n========================================")
print(" BEST MODEL")
print("========================================")

print(
    "Best Model:",
    best_result["model"]
)

print(
    "F1 Score:",
    round(best_result["f1"], 4)
)


# ============================================================
# 21. SHOW SAMPLE PREDICTIONS
# ============================================================

print("\n========================================")
print(" SAMPLE PREDICTIONS")
print("========================================")

best_predictions = {
    "Logistic Regression":
        lr_predictions,

    "Naive Bayes":
        nb_predictions,

    "Random Forest":
        rf_predictions
}[best_result["model"]]


best_predictions.select(
    "text",
    "sentiment",
    "prediction",
    "probability"
).show(
    20,
    truncate=False
)


# ============================================================
# 22. SAVE PREPROCESSING MODEL
# ============================================================

print("\nSaving preprocessing model...")

preprocessing_model.write().overwrite().save(
    "models/twitter_preprocessing"
)


# ============================================================
# 23. SAVE BEST CLASSIFICATION MODEL
# ============================================================

print("Saving best classification model...")

if best_result["model"] == "Logistic Regression":

    lr_model.write().overwrite().save(
        "models/twitter_logistic_regression"
    )

elif best_result["model"] == "Naive Bayes":

    nb_model.write().overwrite().save(
        "models/twitter_naive_bayes"
    )

else:

    rf_model.write().overwrite().save(
        "models/twitter_random_forest"
    )


# ============================================================
# 24. SAVE TEST PREDICTIONS
# ============================================================

print("Saving predictions...")

best_predictions.select(
    "text",
    "sentiment",
    "prediction",
    "probability"
).write.mode(
    "overwrite"
).option(
    "header",
    True
).csv(
    "output/predictions"
)


# ============================================================
# 25. FINAL MESSAGE
# ============================================================

print("\n========================================")
print(" PROJECT PHASE 1 COMPLETED")
print("========================================")

print("""
Completed:

✓ Dataset loading
✓ Missing value handling
✓ Duplicate removal
✓ Text cleaning
✓ Tokenization
✓ Stopword removal
✓ TF-IDF feature extraction
✓ Train/test split
✓ Logistic Regression
✓ Naive Bayes
✓ Random Forest
✓ Accuracy evaluation
✓ Precision evaluation
✓ Recall evaluation
✓ F1 evaluation
✓ Model comparison
✓ Best model selection
✓ Model saving
✓ Prediction output

Next phase:
Build the Web Application and Cloud Deployment.
""")

spark.stop()


 TWITTER SENTIMENT ANALYSIS

[1] Loading dataset...

Dataset Schema:
root
 |-- textID: string (nullable = true)
 |-- text: string (nullable = true)
 |-- selected_text: string (nullable = true)
 |-- sentiment: string (nullable = true)


First 5 rows:
+----------+---------------------------------------------------------------------------+-----------------------------------+---------+
|textID    |text                                                                       |selected_text                      |sentiment|
+----------+---------------------------------------------------------------------------+-----------------------------------+---------+
|cb774db0d1| I`d have responded, if I were going                                       |I`d have responded, if I were going|neutral  |
|549e992a42| Sooo SAD I will miss you here in San Diego!!!                             |Sooo SAD                           |negative |
|088c60f138|my boss is bullying me...                                     

26/08/24 21:22:38 WARN StopWordsRemover: Default locale set was [en_MM]; however, it was not found in available locales in JVM, falling back to en_US locale. Set param `locale` in order to respect another locale.



Processed training data:
+---------+-----+----------------------------------------------------------------------------------------------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|sentiment|label|cleaned_text                                                                                              |features                                                                                                                                                                                                                                                                                     |
+---------+-----+----------------------------------------------------------------------------------------------------------+----------------------------

26/08/24 21:22:47 WARN MemoryStore: Not enough space to cache rdd_949_0 in memory! (computed 195.3 MiB so far)
26/08/24 21:22:47 WARN BlockManager: Persisting block rdd_949_0 to disk instead.
26/08/24 21:22:47 WARN MemoryStore: Not enough space to cache rdd_949_1 in memory! (computed 195.3 MiB so far)
26/08/24 21:22:47 WARN BlockManager: Persisting block rdd_949_1 to disk instead.
26/08/24 21:22:48 WARN MemoryStore: Not enough space to cache rdd_949_0 in memory! (computed 293.1 MiB so far)
26/08/24 21:22:48 WARN MemoryStore: Not enough space to cache rdd_949_1 in memory! (computed 293.1 MiB so far)
26/08/24 21:22:49 WARN MemoryStore: Not enough space to cache rdd_949_1 in memory! (computed 195.3 MiB so far)
26/08/24 21:22:49 WARN MemoryStore: Not enough space to cache rdd_949_0 in memory! (computed 195.3 MiB so far)
26/08/24 21:22:50 WARN MemoryStore: Not enough space to cache rdd_949_1 in memory! (computed 195.3 MiB so far)
26/08/24 21:22:50 WARN MemoryStore: Not enough space to cache


 MODEL EVALUATION

----------------------------------------
Logistic Regression
----------------------------------------
Accuracy  : 0.5539
Precision : 0.5538
Recall    : 0.5539
F1 Score  : 0.5538

----------------------------------------
Naive Bayes
----------------------------------------
Accuracy  : 0.5589
Precision : 0.5587
Recall    : 0.5589
F1 Score  : 0.5577

----------------------------------------
Random Forest
----------------------------------------
Accuracy  : 0.4512
Precision : 0.6773
Recall    : 0.4512
F1 Score  : 0.3333

 MODEL COMPARISON
Logistic Regression       Accuracy=0.5539 F1=0.5538
Naive Bayes               Accuracy=0.5589 F1=0.5577
Random Forest             Accuracy=0.4512 F1=0.3333

 BEST MODEL
Best Model: Naive Bayes
F1 Score: 0.5577

 SAMPLE PREDICTIONS
+-------------------------------------------------------------------------------------------------------------------------------+---------+----------+----------------------------------------------------------

AnalysisException: [UNSUPPORTED_DATA_TYPE_FOR_DATASOURCE] The CSV datasource doesn't support the column `probability` of the type UDT("STRUCT<type: TINYINT NOT NULL, size: INT, indices: ARRAY<INT>, values: ARRAY<DOUBLE>>"). SQLSTATE: 0A000